# CoffeeLeafVision — Evaluation & Comparison

Compara las 4 arquitecturas entrenadas en `02_training.ipynb`. Produce las figuras que alimentan el README, el dashboard y el reporte final.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

with open("../checkpoints/results.json") as f:
    results = json.load(f)

print("Arquitecturas:", list(results.keys()))

## 1. Tabla resumen (media +- std sobre 5 folds)

In [ ]:
rows = []
for arch, data in results.items():
    cv = data["cv_summary"]
    rows.append({
        "Arquitectura": arch,
        "Accuracy": f"{cv['accuracy']['mean']:.4f} +- {cv['accuracy']['std']:.4f}",
        "F1 macro": f"{cv['f1_macro']['mean']:.4f} +- {cv['f1_macro']['std']:.4f}",
        "F1 weighted": f"{cv['f1_weighted']['mean']:.4f} +- {cv['f1_weighted']['std']:.4f}",
        "AUC macro": f"{cv['auc_macro_ovr']['mean']:.4f} +- {cv['auc_macro_ovr']['std']:.4f}",
        "Parametros": f"{data['n_params']:,}",
        "Tamano (MB)": f"{data['size_mb']:.1f}",
        "Latencia CPU (ms)": f"{data['latency_cpu']['median_ms']:.1f}",
    })
summary = pd.DataFrame(rows)
summary

## 2. Curvas de accuracy y F1 por fold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
arch_names = list(results.keys())
for arch in arch_names:
    fold_metrics = results[arch]["fold_metrics"]
    accs = [m["accuracy"] for m in fold_metrics]
    f1s = [m["f1_macro"] for m in fold_metrics]
    axes[0].plot(range(1, len(accs) + 1), accs, marker="o", label=arch)
    axes[1].plot(range(1, len(f1s) + 1), f1s, marker="o", label=arch)
axes[0].set_title("Accuracy por fold")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[1].set_title("F1 macro por fold")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("F1 macro")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Trade-off accuracy vs latencia

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for arch, data in results.items():
    acc = data["cv_summary"]["accuracy"]["mean"]
    lat = data["latency_cpu"]["median_ms"]
    size = data["size_mb"]
    ax.scatter(lat, acc, s=size * 5, alpha=0.6, label=arch)
    ax.annotate(arch, (lat, acc), xytext=(7, 5), textcoords="offset points", fontsize=10)
ax.set_xlabel("Latencia mediana CPU (ms)")
ax.set_ylabel("Accuracy media (5-fold CV)")
ax.set_title("Accuracy vs latencia (tamano de burbuja = MB del modelo)")
plt.tight_layout()
plt.show()

**Interpretacion:**

- Esquina superior izquierda = mejor trade-off (alta accuracy, baja latencia).
- Tamano de burbuja muestra peso del modelo en MB.

## 4. Confusion matrices por arquitectura

In [ ]:
from config import CLASS_LABELS

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, (arch, data) in zip(axes.flat, results.items()):
    cms = np.array([m["confusion_matrix"] for m in data["fold_metrics"]])
    cm_mean = cms.mean(axis=0)
    sns.heatmap(
        cm_mean, annot=True, fmt=".1f", cmap="Blues",
        xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS,
        ax=ax, cbar=False,
    )
    ax.set_title(f"{arch} — confusion matrix promedio (5 folds)")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

## 5. F1 por clase comparativo

In [ ]:
data_rows = []
for arch in arch_names:
    f1_per_class_lists = {label: [] for label in CLASS_LABELS}
    for m in results[arch]["fold_metrics"]:
        for label, val in m["f1_per_class"].items():
            f1_per_class_lists[label].append(val)
    for label in CLASS_LABELS:
        data_rows.append({
            "Arquitectura": arch,
            "Clase": label,
            "F1 mean": np.mean(f1_per_class_lists[label]),
            "F1 std": np.std(f1_per_class_lists[label]),
        })

f1_df = pd.DataFrame(data_rows)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=f1_df, x="Clase", y="F1 mean", hue="Arquitectura", ax=ax)
ax.set_title("F1-score por clase (media de 5 folds)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. Seleccion del modelo de produccion

Criterios:

1. **F1 macro** (metrica principal — penaliza clases minoritarias).
2. **Estabilidad** — preferir modelo con std bajo entre folds.
3. **Latencia** — empate tecnico se rompe a favor del modelo mas rapido.

In [ ]:
best_arch = max(arch_names, key=lambda a: results[a]["cv_summary"]["f1_macro"]["mean"])
best_cv = results[best_arch]["cv_summary"]
print(f"Ganador: {best_arch}")
print(f"  Accuracy: {best_cv['accuracy']['mean']:.4f} +- {best_cv['accuracy']['std']:.4f}")
print(f"  F1 macro: {best_cv['f1_macro']['mean']:.4f} +- {best_cv['f1_macro']['std']:.4f}")
print(f"  AUC macro: {best_cv['auc_macro_ovr']['mean']:.4f} +- {best_cv['auc_macro_ovr']['std']:.4f}")